# Мини-ассистент по Правилам классификации и постройки морских судов (РС) — демо в Colab

Конвейер «LLM + инструменты»: ассистент находит в архиве частей Правил (20 zip-архивов,
оглаавление + страницы) нужные разделы, читает их и отвечает со ссылками на источники
`[doc_id:N, page:M]`.

**Как запустить:**
1. https://colab.research.google.com → **Файл → Загрузить блокнот** → этот файл.
2. Панель **🔑 Секреты** слева → добавьте секреты (имена — в Блоке 0b / Блоке 1).
3. **Среда выполнения → Выполнить все**.

Код разбит на блоки-алгоритмы, каждый в своей ячейке с описанием:
данные → модель оглавления → навигация → архив → промпты → оркестрация из 3 стадий → демо.


## Блок 0a. Установка зависимостей

Единственная зависимость — `openai` (клиент OpenAI-совместимых API: официальный API,
OpenRouter, локальные серверы).

In [ ]:
%pip install -q openai


## Блок 0b. Секреты Colab

Ключи берутся из секретов Colab (панель **🔑 Секреты** → «Add new secret»).
Имена совпадают с локальным `.env.example`:

| Секрет | Назначение |
|---|---|
| `OPENAI_API_KEY` | ключ API (обязателен, либо задать `OPENAI_BASE_URL` без ключа) |
| `OPENAI_BASE_URL` | адрес OpenAI-совместимого сервера (пусто — официальный API) |
| `OPENAI_MODEL` | модель (по умолчанию `gpt-4o-mini`) |
| `LLM_PROVIDER_ONLY` | ограничение провайдера маршрутизации, JSON-массив (опционально) |

`userdata` есть только в Colab — при локальном запуске этой тетради значения
берутся из переменных окружения (или `.env` рядом с ноутбуком).

In [ ]:
import os

try:
    from google.colab import userdata  # доступно только в Colab
except ImportError:
    userdata = None


def _secret(name: str, default: str = "") -> str:
    """Значение из секретов Colab; фолбэк — переменная окружения."""
    if userdata is not None:
        try:
            return userdata.get(name) or default
        except Exception:
            pass
    return os.getenv(name, default)


OPENAI_API_KEY = _secret("OPENAI_API_KEY")
OPENAI_BASE_URL = _secret("OPENAI_BASE_URL")
OPENAI_MODEL = _secret("OPENAI_MODEL") or "gpt-4o-mini"
LLM_PROVIDER_ONLY = _secret("LLM_PROVIDER_ONLY")

assert OPENAI_API_KEY or OPENAI_BASE_URL, (
    "Не задан ключ API. Добавьте секрет OPENAI_API_KEY (панель «🔑 Секреты» слева), "
    "или для локального сервера — OPENAI_BASE_URL (ключ тогда не нужен)."
)
print("Секреты загружены.")
print("  модель:", OPENAI_MODEL)
print("  base_url:", OPENAI_BASE_URL or "(стандартный API)")
print("  провайдер (only):", LLM_PROVIDER_ONLY or "(не ограничен)")


## Блок 1. Загрузка данных: архивы частей Правил

Документы — zip-архивы в публичном репозитории GitHub, вид `2-020101-174-N.zip`
(части I–XX «Правила классификации и постройки морских судов»). Внутри архива:

- `toc.json` — оглавление: дерево заголовков с номерами пунктов (`clause`, напр. 1.1.2)
  и диапазонами страниц;
- `metadata.json` — код и название документа;
- `pages/NNN.md` — текст страниц.

Скачиваем архивы в каталог `data/` (уже загруженные пропускаются).
Если репозиторий сменил ветку/относительный путь — поправить `GITHUB_RAW`.

In [ ]:
import urllib.parse
import urllib.request
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

GITHUB_RAW = "https://raw.githubusercontent.com/neirokoder/search_by_toc/main/data"
ZIP_FILES = ['2-020101-174-1.zip', '2-020101-174-2.zip', '2-020101-174-3.zip', '2-020101-174-4.zip', '2-020101-174-5.zip', '2-020101-174-6.zip', '2-020101-174-7.zip', '2-020101-174-8.zip', '2-020101-174-9.zip', '2-020101-174-10.zip', '2-020101-174-11.zip', '2-020101-174-12.zip', '2-020101-174-13.zip', '2-020101-174-14.zip', '2-020101-174-15.zip', '2-020101-174-16.zip', '2-020101-174-17.zip', '2-020101-174-R-E-18.zip', '2-020101-174-19.zip', '2-020101-174-20.zip']


def download_archives(force: bool = False) -> list[Path]:
    paths = []
    for name in ZIP_FILES:
        path = DATA_DIR / name
        if path.exists() and path.stat().st_size > 0 and not force:
            paths.append(path)
            continue
        url = f"{GITHUB_RAW}/{urllib.parse.quote(name)}"
        for attempt in range(3):
            try:
                urllib.request.urlretrieve(url, path)
                print(f"  [OK] {name} ({path.stat().st_size // 1024} КБ)")
                paths.append(path)
                break
            except Exception as exc:
                if attempt == 2:
                    raise
                print(f"  [!] повтор {attempt + 1} для {name}: {exc}")
    print(f"Архивов в каталоге {DATA_DIR.absolute()}: {len(paths)}")
    return paths


download_archives()


## Блок 2. Модель данных: оглавление и страницы

Алгоритмы подготовки дерева оглавления (`toc.json`):

- **дедупликация** `_dedup_nodes` — соседние узлы с одинаковыми заголовком и уровнем
  сливаются в один (диапазон страниц расширяется, дети объединяются через `_merge_children`);
  компенсирует разобщённую разметку исходных оглавлений;
- **нумерация** `_assign_ids` — сквозные ID узлов;
- **уплощение** `_flatten` — плоский список узлов для поиска по номеру пункта;
- **родители** `_walk_parents` — словарь «узел → родитель» для определения раздела,
  в котором находится пункт (номер пункта может повторяться в разных разделах).

In [ ]:
from dataclasses import dataclass, field


@dataclass
class TocNode:
    """Узел оглавления: заголовок, номер пункта (clause, напр. 1.1.2), уровень,
    диапазон страниц, ID и дочерние узлы."""
    title: str
    clause: str
    level: int
    page_start: int
    page_end: int
    node_id: int = 0
    children: list["TocNode"] = field(default_factory=list)


def _dedup_nodes(nodes: list[dict]) -> list[TocNode]:
    """Сворачивает дубликаты заголовков: соседние узлы с одинаковыми title+level
    сливаются в один (диапазон страниц расширяется, дети объединяются)."""
    out: list[TocNode] = []
    for n in nodes:
        title = (n.get("title") or "").strip()
        clause = (n.get("clause") or "").strip()
        level = int(n.get("level") or 1)
        start = int(n.get("page_start") or 0)
        end = int(n.get("page_end") or 0)
        children = _dedup_nodes(n.get("children") or [])
        if out and out[-1].title == title and out[-1].level == level:
            prev = out[-1]
            prev.page_start = min(prev.page_start, start)
            prev.page_end = max(prev.page_end, end)
            prev.children = _merge_children(prev.children, children)
        else:
            out.append(TocNode(title=title, clause=clause, level=level, page_start=start,
                               page_end=max(end, start), children=children))
    return out


def _merge_children(a: list[TocNode], b: list[TocNode]) -> list[TocNode]:
    """Слияние списков детей при дедупликации родительских узлов."""
    if not b:
        return a
    if not a:
        return b
    if a[-1].title == b[0].title and a[-1].level == b[0].level:
        a[-1].page_start = min(a[-1].page_start, b[0].page_start)
        a[-1].page_end = max(a[-1].page_end, b[0].page_end)
        a[-1].children = _merge_children(a[-1].children, b[0].children)
        return a[:-1] + [a[-1]] + b[1:]
    return a + b


def _assign_ids(nodes: list[TocNode], counter: list[int]) -> None:
    """Сквозная нумерация узлов дерева (для ссылок и поиска родителей)."""
    for n in nodes:
        counter[0] += 1
        n.node_id = counter[0]
        _assign_ids(n.children, counter)


def _flatten(nodes: list[TocNode]) -> list[TocNode]:
    """Плоский список узлов (обход в глубину) для поиска по номерам пунктов."""
    out: list[TocNode] = []
    for n in nodes:
        out.append(n)
        out.extend(_flatten(n.children))
    return out


def _walk_parents(nodes: list[TocNode], stack: list[TocNode | None], parents: dict[int, TocNode]) -> None:
    """Заполняет parents: node_id -> родительский узел."""
    for n in nodes:
        stack.append(n)
        for c in n.children:
            parents[c.node_id] = n
            _walk_parents([c], stack, parents)
        stack.pop()


## Блок 3. Документ: навигация по содержанию и поиск пунктов

Алгоритмы работы с одним документом:

- `navigate(clause, depth, max_nodes)` — навигация по содержанию. Без `clause` — дерево
  с верхнего уровня (или с раскрытием до `depth` уровней, лимит строк `max_nodes`);
  с `clause` — раскрытие конкретного пункта; каждое совпадение выводится внутри своего
  раздела (заголовок раздела сверху), повторные номера пунктов не теряются;
- **fallback по тексту страниц** `_clause_span_in_pages` — если пункта нет в `toc.json`
  (неполная разметка), номер пункта ищется в НАЧАЛЕ абзацев страниц (заголовок пункта,
  а не ссылки-упоминания); диапазон строится до следующего заголовка пункта;
- `pages_by_clause(clauses)` — текст страниц по номерам пунктов (макс. 3 стр. на пункт),
  каждое совпадение — с указанием раздела;
- `get_pages(pages)` — текст страниц по номерам (лимит задаёт вызывающий код, 30).

In [ ]:
import json
import re
import zipfile
from pathlib import Path

_PER_CLAUSE_MAX_PAGES = 3  # сколько страниц показывать на пункт в pages_by_clause
_PART_TITLE_RE = re.compile(r"«([^»]+)»")


class Document:
    """Доступ к архиву одного документа (zip): метаданные, дерево toc, страницы."""

    def __init__(self, number: int, zip_path: Path):
        self.number = number
        self.zip_path = zip_path
        self._meta: dict | None = None
        self._tree: list[TocNode] | None = None
        self._flat: list[TocNode] | None = None
        self._pages_raw: int | None = None
        self._parent: dict[int, TocNode] | None = None

    def __repr__(self) -> str:
        return f"Document({self.number}, {self.doc_code})"

    @property
    def meta(self) -> dict:
        if self._meta is None:
            with zipfile.ZipFile(self.zip_path) as zf:
                self._meta = json.loads(zf.read("metadata.json"))
        return self._meta

    @property
    def doc_code(self) -> str:
        return self.meta.get("doc_code") or self.zip_path.stem

    @property
    def title(self) -> str:
        """Короткое название части (текст в кавычках из metadata.title)."""
        raw = self.meta.get("title") or ""
        m = _PART_TITLE_RE.search(raw)
        return m.group(1).strip() if m else raw[:120].strip()

    @property
    def part_label(self) -> str:
        """Заголовок книги: 'часть I «Классификация»' из metadata.title."""
        raw = self.meta.get("title") or ""
        m = re.search(r"части\s+([IVXLCDM]+)\s*«([^»]+)»", raw, re.IGNORECASE)
        if m:
            return f"часть {m.group(1)} «{m.group(2).strip()}»"
        return self.title

    @property
    def pages_count(self) -> int:
        return self._pages_count_raw or max(p.page_end for p in self.flat_nodes())

    @property
    def _pages_count_raw(self) -> int:
        if self._pages_raw is None:
            with zipfile.ZipFile(self.zip_path) as zf:
                toc = json.loads(zf.read("toc.json"))
            self._pages_raw = int(toc.get("pages_count") or 0)
        return self._pages_raw

    def tree(self) -> list[TocNode]:
        """Дерево оглавления с дедупликацией, ID и картой родителей."""
        if self._tree is None:
            with zipfile.ZipFile(self.zip_path) as zf:
                toc = json.loads(zf.read("toc.json"))
            self._tree = _dedup_nodes(toc.get("tree") or [])
            _assign_ids(self._tree, [0])
            parents: dict[int, TocNode] = {}
            stack = [None]
            _walk_parents(self._tree, stack, parents)
            self._parent = parents
        return self._tree

    def flat_nodes(self) -> list[TocNode]:
        if self._flat is None:
            self._flat = _flatten(self.tree())
        return self._flat

    def _section_path(self, node: TocNode) -> str:
        """Название раздела (уровень 1), в котором находится узел."""
        parts = []
        cur = node
        seen = 0
        while cur is not None and seen < 100:
            if cur.level == 1:
                parts.append(cur.title)
                break
            cur = self._parent.get(cur.node_id)
            seen += 1
        return parts[0] if parts else ""

    def _find_clause(self, clause: str) -> list[TocNode]:
        clause = clause.rstrip(".")
        return [n for n in self.flat_nodes() if n.clause == clause]

    def _find_clause_in_pages(self, clause: str) -> list[tuple[int, str]]:
        """Поиск номера пункта в начале абзацев текста страниц, когда пункт
        отсутствует в оглавлении (неполная разметка toc). НЕ ищет упоминания-ссылки."""
        clause = clause.rstrip(".")
        pat = re.compile(rf"(?m)^\s*(?:\*\s*)?{re.escape(clause)}(?![\d.])([^\n]*)")
        hits: list[tuple[int, str]] = []
        with zipfile.ZipFile(self.zip_path) as zf:
            names = sorted(n for n in zf.namelist() if n.startswith("pages/") and n.endswith(".md"))
            for name in names:
                page = int(name[len("pages/"):-len(".md")])
                txt = zf.read(name).decode("utf-8", errors="replace")
                m = pat.search(txt)
                if m:
                    hits.append((page, m.group(1).strip(" -")))
        return hits

    def _clause_span_in_pages(self, clause: str) -> tuple[int, int, str] | None:
        """Диапазон страниц пункта, найденного в тексте (fallback при неполном
        оглавлении): от страницы с заголовком пункта до страницы перед
        следующим заголовком пункта. Возвращает (первая, последняя, заголовок)."""
        hits = self._find_clause_in_pages(clause)
        if not hits:
            return None
        start = hits[0][0]
        title = hits[0][1]
        end = start
        next_head = re.compile(r"(?m)^\s*(?:\*\s*)?(\d+(?:\.\d+)+)(?![\d.])([^\n]*)")
        with zipfile.ZipFile(self.zip_path) as zf:
            names = sorted(n for n in zf.namelist() if n.startswith("pages/") and n.endswith(".md"))
            for name in names:
                page = int(name[len("pages/"):-len(".md")])
                if page <= start:
                    continue
                if page > end + 1:
                    break
                txt = zf.read(name).decode("utf-8", errors="replace")
                m = next_head.search(txt)
                if m and not m.group(1).startswith(clause + "."):
                    break
                end = page
        return start, end, title

    def get_page(self, page: int) -> str | None:
        name = f"pages/{page:03d}.md"
        with zipfile.ZipFile(self.zip_path) as zf:
            if name not in zf.namelist():
                return None
            return zf.read(name).decode("utf-8", errors="replace")

    def get_pages(self, pages: list[int]) -> str:
        """Форматированный вывод страниц: текст каждой страницы с пометкой."""
        parts = []
        for p in pages:
            content = self.get_page(p)
            if content is None:
                parts.append(f"===== Документ №{self.number}, стр. {p}: страница не найдена =====")
            else:
                parts.append(f"===== Документ №{self.number}, стр. {p} =====\n{content}")
        return "\n\n".join(parts)

    def _section_root(self, node: TocNode) -> TocNode | None:
        """Узел раздела (уровень 1), в котором находится узел, или None."""
        cur = node
        seen = 0
        while cur is not None and seen < 100:
            if cur.level == 1:
                return cur
            cur = self._parent.get(cur.node_id)
            seen += 1
        return None

    def _format_clause_matches(self, matches: list[TocNode], clause: str,
                               depth: int, max_nodes: int,
                               counter: list[int] | None) -> list[str]:
        """Строки для найденных пунктов: каждый пункт выводится внутри своего
        раздела (заголовок раздела один на группу), а не с пометкой на строке."""
        groups: dict[int, list[TocNode]] = {}
        for n in matches:
            root = self._section_root(n)
            key = root.node_id if root else id(n)
            groups.setdefault(key, []).append(n)
        lines: list[str] = []
        for nodes in groups.values():
            if counter is not None and counter[0] >= max_nodes:
                counter[1] = True
                break
            root = self._section_root(nodes[0])
            if root and not (len(nodes) == 1 and nodes[0] is root):
                lines.append(self._format_node(root))
                if counter is not None:
                    counter[0] += 1
            for n in nodes:
                if counter is not None and counter[0] >= max_nodes:
                    counter[1] = True
                    break
                indent = "  " if root and not (len(nodes) == 1 and nodes[0] is root) else ""
                lines.append(indent + self._format_node(n))
                if counter is not None:
                    counter[0] += 1
                if n.children and depth > 1:
                    lines.extend(self._format_levels(n.children, depth - 1, max_nodes,
                                                     indent=indent + "  ", counter=counter))
        return lines

    def navigate(self, clause: str | None = None, depth: int = 1, max_nodes: int = 300) -> str:
        """Форматированное содержание. clause — номер пункта для раскрытия
        (без него — с верхнего уровня); depth — сколько уровней вложенности
        показать (по умолчанию 1, выводится полностью; лимит max_nodes — только
        при depth > 1). Номер пункта может повторяться в разных разделах."""
        counter = [0, False] if depth > 1 else None
        if not clause:
            prefix = f"Документ №{self.number} — {self.part_label} ({self.pages_count} стр.).\n"
            lines = self._format_levels(self.tree(), depth, max_nodes, counter=counter)
            if counter and counter[1]:
                lines.append(f"... и ещё строк (лимит {max_nodes}); уточни пункт или запроси меньшую глубину")
            return prefix + "\n".join(lines)
        matches = self._find_clause(clause.strip())
        if not matches:
            span = self._clause_span_in_pages(clause.strip())
            if span:
                start, end, title = span
                rng = f"стр. {start}–{end}" if end > start else f"стр. {start}"
                return f"Документ №{self.number} — {self.part_label} ({self.pages_count} стр.).\n{clause} {title} — {rng}"
            return (f"Документ №{self.number} — {self.part_label} ({self.pages_count} стр.).\n"
                    f"Пункт {clause} не найден в содержании документа №{self.number}. "
                    "Вызови навигацию без clause для просмотра верхнего уровня.")
        lines: list[str] = []
        if len(matches) == 1 and matches[0].level == 1 and depth == 1:
            lines.append(self._format_node(matches[0]))
        else:
            lines = self._format_clause_matches(matches, clause, depth, max_nodes, counter)
        if counter and counter[1]:
            lines.append(f"... и ещё строк (лимит {max_nodes}); уточни пункт или запроси меньшую глубину")
        return f"Документ №{self.number} — {self.part_label} ({self.pages_count} стр.).\n" + "\n".join(lines)

    @staticmethod
    def _format_levels(nodes: list[TocNode], depth: int, max_nodes: int,
                       indent: str = "", counter: list[int] | None = None) -> list[str]:
        """Строки дерева до глубины depth. При depth == 1 (counter=None)
        вывод полный; при вложенных уровнях — общий лимит max_nodes строк."""
        lines: list[str] = []
        for n in nodes:
            if counter is not None and counter[0] >= max_nodes:
                counter[1] = True
                break
            lines.append(indent + Document._format_node(n))
            if counter is not None:
                counter[0] += 1
            if n.children and depth > 1:
                lines.extend(Document._format_levels(n.children, depth - 1, max_nodes,
                                                     indent + "  ", counter))
        return lines

    @staticmethod
    def _format_node(n: TocNode) -> str:
        rng = f"{n.page_start}–{n.page_end}" if n.page_end > n.page_start else str(n.page_start)
        clause = f"{n.clause} " if n.clause else ""
        marker = " ▸" if n.children else ""
        return f"{clause}{n.title} — стр. {rng}{marker}"

    def pages_by_clause(self, clauses: list[str]) -> str:
        """Страницы, содержащие пункты с указанными номерами (clause).
        Номера могут повторяться в разных разделах — каждое совпадение выводится
        отдельно с указанием раздела. На пункт — не более _PER_CLAUSE_MAX_PAGES
        страниц. Возвращает описание найденных пунктов и текст их страниц."""
        clauses = [c.strip() for c in clauses if c and c.strip()]
        if not clauses:
            return "Не указаны номера пунктов (clauses)."
        header: list[str] = []
        pages: set[int] = set()
        for c in clauses:
            matches = self._find_clause(c)
            if not matches:
                span = self._clause_span_in_pages(c)
                if span:
                    start, end, title = span
                    shown_end = min(end, start + _PER_CLAUSE_MAX_PAGES - 1)
                    rng = f"{start}–{shown_end}" if shown_end > start else str(start)
                    note = f"; всего стр. {end - start + 1}, показаны первые {_PER_CLAUSE_MAX_PAGES}" if end - start + 1 > _PER_CLAUSE_MAX_PAGES else ""
                    header.append(f"{c} «{title}» — стр. {rng}{note}")
                    pages.update(range(start, shown_end + 1))
                    continue
                header.append(f"Пункт {c} не найден в содержании документа №{self.number}.")
                continue
            for n in matches:
                span = n.page_end - n.page_start + 1
                shown_end = min(n.page_end, n.page_start + _PER_CLAUSE_MAX_PAGES - 1)
                rng = f"{n.page_start}–{shown_end}" if shown_end > n.page_start else str(n.page_start)
                section = self._section_path(n)
                where = f" (раздел «{section}»)" if section else ""
                note = f"; всего стр. {span}, показаны первые {_PER_CLAUSE_MAX_PAGES}" if span > _PER_CLAUSE_MAX_PAGES else ""
                header.append(f"{c} «{n.title}» — стр. {rng}{where}{note}")
                pages.update(range(n.page_start, shown_end + 1))
        if not pages:
            return "\n".join(header)
        body = self.get_pages(sorted(pages))
        return "\n".join(header) + "\n\n" + body


## Блок 4. Архив: перечень документов

`Archive` сканирует каталог на zip-архивы с номером в имени (`-N.zip`, паттерн регэкспа),
сортирует по номеру и предоставляет:

- `documents()` / `get(number)` — список документов / доступ по номеру (кэш);
- `list_text(query, doc_code)` — перечень строк «№N — код «название» (стр.)» с фильтрами:
  `query` — все ключевые слова из названия, `doc_code` — подстрока кода документа.

In [ ]:
_NUMBER_RE = re.compile(r"-(\d+)\.zip$")


class Archive:
    """Каталог zip-архивов документов: перечень, доступ по номеру, фильтры."""

    def __init__(self, data_dir: Path):
        self.data_dir = data_dir
        self._docs: dict[int, Document] = {}

    def documents(self) -> list[Document]:
        if not self._docs:
            docs = []
            for p in sorted(self.data_dir.glob("*.zip")):
                m = _NUMBER_RE.search(p.name)
                if m:
                    docs.append(Document(int(m.group(1)), p))
            docs.sort(key=lambda d: d.number)
            self._docs = {d.number: d for d in docs}
        return list(self._docs.values())

    def get(self, number: int) -> Document | None:
        self.documents()
        return self._docs.get(number)

    def list_text(self, query: str | None = None, doc_code: str | None = None) -> str:
        """Перечень документов. query — ключевые слова по названию (все слова),
        doc_code — подстрока кода документа (например 174-1)."""
        docs = self.documents()
        q = (query or "").strip().lower()
        code = (doc_code or "").strip().lower()
        if q or code:
            docs = [
                d for d in docs
                if (not q or all(w in (d.title + " " + d.doc_code).lower() for w in q.split()))
                and (not code or code in (d.doc_code + " " + d.zip_path.stem).lower())
            ]
        if not docs:
            return "Документы по запросу не найдены."
        lines = []
        for d in docs:
            lines.append(f"№{d.number} — {d.doc_code} «{d.title}» ({d.pages_count} стр.)")
        return "\n".join(lines)


## Блок 5. Промпты и определения инструментов

Ассистент общается с архивом через 4 инструмента (OpenAI function calling):

| Инструмент | Назначение |
|---|---|
| `list_documents` | перечень документов; фильтры `query`, `doc_code` |
| `toc_navigate` | навигация по содержанию; `requests` — массив запросов (несколько документов/пунктов за один вызов), `clause` — пункт для раскрытия, `depth` — уровни |
| `get_pages_by_clause` | страницы пунктов; `requests` — массив (документ + `clauses`) |
| `get_page_markdown` | страницы по номерам (макс. 30 за вызов) |

Системный промпт задаёт двухшаговую схему (черновик → план проверки и сверка),
правила выбора инструмента по типу запроса и обязательные ссылки на источники
`[doc_id:N, page:M]` после каждого утверждения.

In [ ]:
_LIST_DOCUMENTS_TOOL = {
    "type": "function",
    "function": {
        "name": "list_documents",
        "description": (
            "Список доступных документов архива: номер, код, название, число страниц. "
            "Можно отфильтровать по ключевым словам названия (query) или коду (doc_code). "
            "Вызывай когда пользователь называет документ неточно или нужно узнать состав архива."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Ключевые слова из названия (все слова должны встречаться), например 'корпус'",
                },
                "doc_code": {
                    "type": "string",
                    "description": "Подстрока кода документа, например '174-2'",
                },
            },
        },
    },
}

_TOC_NAVIGATE_TOOL = {
    "type": "function",
    "function": {
        "name": "toc_navigate",
        "description": (
            "Навигация по содержанию (оглавлению) документов. "
            "Показывает заголовки с номерами пунктов (например 1.1.2) и страницами. "
            "Параметр requests — список запросов: в каждом укажи document_id и при необходимости "
            "clause (номер пункта для раскрытия, без него — содержание с верхнего уровня) и depth "
            "(сколько уровней вложенности показать, по умолчанию 2, максимум 5). Одним вызовом "
            "покрывается сразу несколько документов и/или пунктов — результаты объединяются. "
            "Каждый результат начинается с заголовка «Документ №N — часть X «Название»». "
            "Номер пункта может повторяться в разных разделах — каждое совпадение выводится внутри своего раздела (заголовок раздела сверху)."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "requests": {
                    "type": "array",
                    "description": "Список запросов по документам",
                    "items": {
                        "type": "object",
                        "properties": {
                            "document_id": {
                                "type": "integer",
                                "description": "Номер документа из перечня в системном промпте",
                            },
                            "clause": {
                                "type": "string",
                                "description": (
                                    "Номер пункта для раскрытия вложенных подразделов, например '1.1' или '1.1.2'. "
                                    "Без него возвращается содержание с верхнего уровня."
                                ),
                            },
                            "depth": {
                                "type": "integer",
                                "description": (
                                    "Сколько уровней вложенности показать: 1 — верхний уровень, "
                                    "2 — верхний и вложенные подразделы (по умолчанию), 3 — ещё глубже."
                                ),
                                "minimum": 1,
                                "maximum": 5,
                            },
                        },
                        "required": ["document_id"],
                    },
                },
            },
            "required": ["requests"],
        },
    },
}

_GET_PAGE_MD_TOOL = {
    "type": "function",
    "function": {
        "name": "get_page_markdown",
        "description": (
            "Загрузить текст страниц документа. Вызывай когда найден нужный раздел "
            "в toc_navigate и необходимо прочитать его содержимое. "
            "Максимум 30 страниц за вызов."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "document_id": {
                    "type": "integer",
                    "description": "Номер документа из перечня в системном промпте",
                },
                "pages": {
                    "type": "array",
                    "items": {"type": "integer"},
                    "description": "Список номеров страниц (1-based) для загрузки",
                },
            },
            "required": ["document_id", "pages"],
        },
    },
}

_GET_PAGES_BY_CLAUSE_TOOL = {
    "type": "function",
    "function": {
        "name": "get_pages_by_clause",
        "description": (
            "Загрузить текст страниц документов, содержащих указанные пункты (номера заголовков, "
            "например 1.1.2). Номера пунктов указываются в содержании (toc_navigate) и используются "
            "для ссылок в тексте документа. Параметр requests — список запросов: в каждом укажи "
            "document_id и номера пунктов clauses (можно несколько). Одним вызовом покрывается "
            "сразу несколько документов и/или пунктов — страницы объединяются."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "requests": {
                    "type": "array",
                    "description": "Список запросов по документам",
                    "items": {
                        "type": "object",
                        "properties": {
                            "document_id": {
                                "type": "integer",
                                "description": "Номер документа из перечня в системном промпте",
                            },
                            "clauses": {
                                "type": "array",
                                "description": "Номера пунктов, например ['1.1.2', '3.4.1']",
                                "items": {"type": "string"},
                            },
                        },
                        "required": ["document_id", "clauses"],
                    },
                },
            },
            "required": ["requests"],
        },
    },
}

_TOOLS = [_LIST_DOCUMENTS_TOOL, _TOC_NAVIGATE_TOOL, _GET_PAGES_BY_CLAUSE_TOOL, _GET_PAGE_MD_TOOL]

_SYSTEM_PROMPT = """
Ты — ассистент по нормативно-техническим документам Российского морского регистра судоходства (РС, Регистр)
с опытом инженера кораблестроения. Ты работаешь с архивом документов, доступ к которым у тебя есть
только через инструменты. У тебя есть следующие инструменты:
- list_documents — список документов архива (номера, коды, названия, страницы).
- toc_navigate — навигация по содержанию документа: заголовки с номерами пунктов (1.1.2), вложенными уровнями и страницами.
- get_pages_by_clause — загрузка страниц документа по номерам пунктов (например 1.1.2).
- get_page_markdown — загрузка текста страниц документа по номерам страниц.

Доступные документы архива (номер — код «название»):
[DOCUMENTS]

ПЛАН ДЕЙСТВИЙ (двухшаговая схема):
ШАГ 1 — по твоему запросу будет дан вопрос и ты ответишь ПОЛНОСТЬЮ на основе своих знаний,
без обращения к инструментам и документам. Это предварительный черновик.
ШАГ 2 – ПЛАН ПРОВЕРКИ:
На основе черновика выдели ключевые термины, номера пунктов, названия разделов, которые упоминаются в черновике. Используй их для:
- формирования query для list_documents (если документ ещё не определён),
- или для выбора clause при вызове toc_navigate (если документ уже известен),
- или для непосредственного вызова get_pages_by_clause с этими номерами пунктов (если они явно указаны в черновике).
- делай обращение к содержанию и страницам пакетно, что бы за минимум запросов получить максимум нужных данных
- экономь контекст, не запрашивай дважды то, что уже получено и есть в кэше
План должен сопровождаться немедленным вызовом инструмента в том же ответе.
Навигацией по содержанию (toc_navigate) найди нужные разделы, загрузи их страницы (get_pages_by_clause, get_page_markdown) и на основании полученного содержимого сверь черновик и собери финальный ответ.
В следующих итерациях повторный план не нужен — просто продолжай выполнение.

ВЫБОР ИНСТРУМЕНТА ПО ТИПУ ЗАПРОСА:
1. Пользователь ссылается на документ по номеру, названию или коду → list_documents (при неточном названии укажи query или doc_code), чтобы уточнить номер.
2. Нужно найти раздел по теме или названию → toc_navigate (сначала без clause — верхний уровень, затем раскрывай пункты ▸ по их номерам).
3. Известен номер пункта (например 1.1.2) — пользователь спросил про конкретный пункт или ссылку из текста → get_pages_by_clause.
4. Известен раздел и номера страниц → get_page_markdown.
5. Если вопрос можно решить на основе истории диалога — отвечай напрямую без вызова инструментов.

ВАЖНО:
- Содержание в системном промпте НЕ передаётся — структура документов доступна только через toc_navigate.
- В содержании всегда указываются номера пунктов (1.1.2) — по ним делаются ссылки в тексте документов.
- Если toc_navigate не дал нужного пункта — посмотри другие ветки содержания, не выдумывай страницы.
- Не более 30 страниц за один вызов get_page_markdown; при необходимости вызывай повторно.
- Черновик по знаниям (шаг 1) — это только материал для проверки: финальный ответ формируй
  исключительно на основе загруженных страниц, исправляя и дополняя черновик фактами из документов.
- Если в документах нет нужной информации — сообщи: «В доступных документах не найдена информация по данному запросу.»
- Не упоминай инструменты и технические детали своей работы в ответе.
- После каждого утверждения ставь ссылку на источник: [doc_id:<номер>, page:<страница>],
  например [doc_id:1, page:5]. Без ссылки на источник не пиши утверждений. Не дублируй одинаковые ссылки подряд, ставь одну по итогу.
- Если для ответа достаточно информации, уже полученной в этой сессии (загруженные страницы, результаты toc_navigate), не вызывай новые инструменты — используй имеющиеся данные.
- Для оформления данных и таблиц используй Markdown и LaTeX. Обязательно форматируй формулы.
- После завершения проверь, насколько корректно выполнена задача и всё ли учтено при ответе.
"""

_DRAFT_PROMPT = (
    "Ответь на вопрос пользователя максимально полно и подробно исключительно на основе "
    "своих знаний с указанием частей документов. Это предварительный "
    "черновик — далее он будет сверен с документами архива, поэтому прямые ссылки здесь отсуствуют [doc_id...]. "
    "В конце проверь, все ли части документов ты учел и на сколько корректен ответ."
)

_VERIFY_PROMPT = (
    "На основании предварительного ответа (черновика, выше в истории):\n"
    "Составь план проверки этого черновика по документам в базе: определи, какие разделы, "
    "пункты и страницы необходимо изучить. Сразу начни выполнение с вызова инструментов: "
    "навигация по содержанию, загрузка страниц. Загружай страницы сразу пакетно. Сверь черновик с документами: подтверди "
    "или исправь каждое утверждение, дополни недостающие детали. Затем подготовь финальный "
    "ответ со ссылками [doc_id:<номер>, page:<страница>] после каждого утверждения."
)

_REVIEW_PROMPT = (
    "Оцени, насколько корректно выполнена задача: учтены ли доступные источники, "
    "детали из загруженных страниц и качество анализа. "
    "Обязательно проверь наличие ссылок [doc_id:N, page:M] после утверждений — текст без ссылок недопустим. "
    "Если есть исправления — выдай обновлённый ответ. "
    "Если исправления не нужны, напиши: 'Ответ корректен, изменений не требуется.' "
    "Не выдавай в финале никаких пояснений, выводов перепроверки и упоминаний о ней — сразу выдавай исправленный ответ или фразу об отсутствии изменений."
    "Не пиши в финале никаких упоминаний о том, что это проверка корректности, для пользователя выдается просто улучшенный скорректированный ответ"
)


## Блок 6. Оркестрация: конвейер из 3 стадий

Алгоритм обработки запроса:

**Стадия 1 — черновик по знаниям.** Модель отвечает полностью на основе своих знаний,
без инструментов и документов (`_DRAFT_PROMPT`). Это материал для последующей сверки.

**Стадия 2 — план и сверка с документами.** Модель составляет план проверки (`_VERIFY_PROMPT`)
и немедленно начинает вызывать инструменты. Цикл `_run_tool_loop` (до 25 итераций):
модель возвращает `tool_calls` → `_call_tool` выполняет их (навигация/страницы) → результаты
попадают в историю сообщений → модель анализирует и продолжает. Ответ текстом принимается
только после фильтров: `_is_garbage` (повторяющийся мусор) и `_PLAN_ONLY_RE` (ответ-план
без фактов) — в таких случаях цикл продолжается с подсказкой.

**Стадия 3 — финализация и контроль качества.**
- `_wind_down` — если лимит итераций исчерпан, ответ принудительно формируется текстом:
  серия запрещающих инструкций с `tool_choice="none"`;
- `_review` — отдельный проход LLM проверяет ответ на корректность и наличие ссылок
  `[doc_id:N, page:M]`, при необходимости выдаёт исправленную версию;
- история диалога хранится и подмешивается к следующим запросам.

Дополнительно в `_complete` — пауза между запросами к API (гуманность к rate limit),
ретраи с экспоненциальной паузой при сбоях и передача ограничения провайдера
маршрутизации `LLM_PROVIDER_ONLY` (например OpenRouter).

In [ ]:
import time
from dataclasses import dataclass

from openai import OpenAI

_MAX_TOOL_ITERS = 25
_MAX_PAGES_PER_CALL = 30

_WIND_DOWN_PROHIBITIONS = [
    "На основе всей полученной информации напиши ответ пользователю. Не вызывай инструменты.",
    "Ты исчерпал лимит вызовов инструментов. ОТВЕТЬ ТЕКСТОМ на основе собранных данных. Не вызывай инструменты.",
    "ЗАПРЕЩЕНО вызывать инструменты. Напиши ответ прямо сейчас.",
    "НЕМЕДЛЕННО ОТВЕТЬ ТЕКСТОМ. Любые вызовы инструментов будут проигнорированы.",
    "ФИНАЛЬНОЕ ПРЕДУПРЕЖДЕНИЕ: ответь текстом на основе того, что уже собрано.",
]

_GENERIC_WORDS = {
    "что", "как", "где", "когда", "зачем", "почему", "какие", "какой", "какая",
    "этот", "эта", "это", "эти", "его", "её", "их", "наш", "ваш",
    "должен", "должна", "должны", "нужно", "необходимо", "может", "могут",
    "является", "являются", "имеет", "имеют", "все", "всё", "также",
    "определение", "понятие", "сущность",
    "рассказать", "расскажи", "покажи", "показать", "показывает",
    "назвать", "назови", "перечислить", "перечисли",
}

_PLAN_ONLY_RE = re.compile(r"^\s*(план|шаг|этап)\b", re.IGNORECASE)


def _is_garbage(text: str) -> bool:
    """Фильтр «мусорного» ответа: пустой, короткий, с зацикленными повторами слов
    или с крайне бедным словарным запасом."""
    if not text:
        return True
    text = text.strip()
    if len(text) < 10:
        return False
    words = text.lower().split()
    if len(words) < 5:
        return False
    for w in set(words):
        run = 0
        for w2 in words:
            if w2 == w:
                run += 1
                if run >= 8:
                    return True
            else:
                run = 0
    if len(words) > 200 and len(set(words)) / len(words) < 0.05:
        return True
    return False


@dataclass
class Settings:
    """Настройки ассистента; LLM-параметры берутся из секретов Colab (Блок 1)."""
    api_key: str
    base_url: str = ""
    model: str = "gpt-4o-mini"
    max_toc_nodes: int = 300
    max_pages_per_call: int = 30
    llm_pause_seconds: float = 1.0
    llm_provider_only: str = ""


class MiniAssistant:
    """Оркестрация: 3 стадии обработки запроса + цикл вызовов инструментов."""

    def __init__(self, archive: Archive, settings: Settings):
        self.archive = archive
        self.settings = settings
        kwargs = {"api_key": settings.api_key or ("sk-local" if settings.base_url else "")}
        if settings.base_url:
            kwargs["base_url"] = settings.base_url
        self.llm = OpenAI(**kwargs)
        self.model = settings.model
        self.history: list[dict] = []
        self._last_llm_at: float | None = None

    def _complete(self, messages: list[dict], tools=None, tool_choice=None, retries: int = 4):
        """Вызов LLM с паузой между запросами и повторами при сбоях/rate limit."""
        now = time.monotonic()
        if self._last_llm_at is not None:
            wait = self.settings.llm_pause_seconds - (now - self._last_llm_at)
            if wait > 0:
                time.sleep(wait)
        self._last_llm_at = time.monotonic()
        extra_body: dict | None = None
        if self.settings.llm_provider_only.strip():
            try:
                extra_body = {"provider": {"only": json.loads(self.settings.llm_provider_only)}}
            except json.JSONDecodeError:
                print("  [!] LLM_PROVIDER_ONLY не является JSON-массивом, поле provider не передаётся")
        for attempt in range(retries + 1):
            try:
                return self.llm.chat.completions.create(
                    model=self.model, messages=messages, tools=tools,
                    tool_choice=tool_choice, extra_body=extra_body,
                )
            except Exception as exc:
                last = attempt == retries
                if last:
                    raise
                wait = 10 * (attempt + 1)
                print(f"  [!] сбой LLM ({type(exc).__name__}: {str(exc)[:120]}) — повтор {attempt + 1}/{retries} через {wait} с", flush=True)
                time.sleep(wait)

    def _system_prompt(self) -> str:
        docs_text = self.archive.list_text()
        return _SYSTEM_PROMPT.replace("[DOCUMENTS]", docs_text)

    def _call_tool(self, name: str, args: dict) -> str:
        """Исполнение одного вызова инструмента (навигация/страницы/перечень)."""
        if name == "list_documents":
            return self.archive.list_text(query=args.get("query"), doc_code=args.get("doc_code"))

        if name == "toc_navigate":
            reqs = args.get("requests") or [{"document_id": args.get("document_id"),
                                              "clause": args.get("clause"),
                                              "depth": args.get("depth")}]
            parts = []
            for req in reqs:
                doc = self.archive.get(int(req.get("document_id")))
                if doc is None:
                    parts.append("Документ с номером %s не найден. Вызови list_documents для получения перечня." % req.get("document_id"))
                    continue
                clause = req.get("clause")
                depth = int(req.get("depth") or 2)
                parts.append(doc.navigate(str(clause).strip() if clause else None,
                                          depth=depth, max_nodes=self.settings.max_toc_nodes))
            return "\n\n".join(parts)

        if name == "get_pages_by_clause":
            reqs = args.get("requests") or [{"document_id": args.get("document_id"),
                                             "clauses": args.get("clauses") or []}]
            parts = []
            for req in reqs:
                doc = self.archive.get(int(req.get("document_id")))
                if doc is None:
                    parts.append("Документ с номером %s не найден. Вызови list_documents для получения перечня." % req.get("document_id"))
                    continue
                clauses = [str(c) for c in (req.get("clauses") or [])]
                parts.append(doc.pages_by_clause(clauses))
            return "\n\n".join(parts)

        if name == "get_page_markdown":
            doc = self.archive.get(int(args.get("document_id")))
            if doc is None:
                return "Документ с таким номером не найден. Вызови list_documents для получения перечня."
            pages = [int(p) for p in (args.get("pages") or [])]
            if not pages:
                return "Не указаны номера страниц (pages)."
            pages = pages[:_MAX_PAGES_PER_CALL]
            return doc.get_pages(pages)

        return f"Неизвестный инструмент: {name}"

    def _run_tool_loop(self, messages: list[dict]) -> str:
        """Стадия 2, цикл: план → tool_calls → исполнение → результаты → анализ."""
        for i in range(_MAX_TOOL_ITERS):
            print(f"  > итерация {i + 1}: анализ...", flush=True)
            response = self._complete(messages, tools=_TOOLS)
            msg = response.choices[0].message
            if msg.content:
                print(f"  > план: {msg.content.strip()[:300]}", flush=True)
            if not msg.tool_calls:
                content = msg.content or ""
                if _is_garbage(content) or _PLAN_ONLY_RE.match(content):
                    messages.append({"role": "user",
                                     "content": "Сгенерирован некорректный ответ без вызова инструментов. Продолжай: вызови нужный инструмент или дай ответ."})
                    continue
                return content
            messages.append({
                "role": "assistant",
                "content": msg.content,
                "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
            })
            for tc in msg.tool_calls:
                fn = tc.function.name
                args = json.loads(tc.function.arguments or "{}")
                print(f"  > вызов {fn}({json.dumps(args, ensure_ascii=False)[:120]})", flush=True)
                tool_content = self._call_tool(fn, args)
                preview = " ".join(tool_content.split())[:400]
                print(f"  > результат: {preview}", flush=True)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": tool_content,
                })
        return ""

    def _wind_down(self, messages: list[dict]) -> str:
        """Стадия 3: принудительная финализация текстом, если итерации исчерпаны."""
        for attempt, prohibition in enumerate(_WIND_DOWN_PROHIBITIONS):
            print(f"  > формирование ответа (попытка {attempt + 1}/{len(_WIND_DOWN_PROHIBITIONS)})...", flush=True)
            response = self._complete(messages, tools=_TOOLS, tool_choice="none")
            content = (response.choices[0].message.content or "").strip()
            if content and not _is_garbage(content) and not _PLAN_ONLY_RE.match(content):
                return content
            messages.append({"role": "user", "content": prohibition})
        return ""

    def _review(self, messages: list[dict], answer: str) -> str:
        """Стадия 3: контроль качества — проверка корректности и наличия ссылок."""
        if "в доступных документах не найдена информация" in answer.lower():
            return answer
        try:
            review_messages = list(messages) + [{"role": "user", "content": _REVIEW_PROMPT}]
            response = self._complete(review_messages, tools=_TOOLS)
            content = (response.choices[0].message.content or "").strip()
            if (content and not _is_garbage(content)
                    and "изменений не требуется" not in content.lower()
                    and "[doc_id:" in content):
                return content
        except Exception as exc:
            print(f"  [!] проверка ответа не удалась ({exc}), оставляю исходный", flush=True)
        return answer

    def run(self, user_query: str) -> str:
        """Обработка запроса: 3 стадии с историей диалога."""
        messages = [{"role": "system", "content": self._system_prompt()}]
        messages.extend(self.history)

        # Стадия 1: черновик по знаниям, без инструментов и документов
        print("> Стадия 1: черновик на основе знаний (без документов)...", flush=True)
        messages.append({"role": "user", "content": _DRAFT_PROMPT + f"\n\nВопрос:\n{user_query}"})
        draft_resp = self._complete(messages, tools=None)
        draft = (draft_resp.choices[0].message.content or "").strip()
        print(f"> Черновик:\n{draft}\n", flush=True)
        messages.append({"role": "assistant", "content": draft})

        # Стадия 2: план и сверка с документами через инструменты
        print("> Стадия 2: план и сверка с документами...", flush=True)
        messages.append({"role": "user", "content": _VERIFY_PROMPT})
        answer = self._run_tool_loop(messages)
        if not answer:
            answer = self._wind_down(messages)
        if not answer:
            answer = "Не удалось найти решение за отведённое время. Попробуйте уточнить запрос."

        # Стадия 3: контроль качества
        answer = self._review(messages, answer)
        self.history.append({"role": "user", "content": user_query})
        self.history.append({"role": "assistant", "content": answer})
        return answer


## Блок 7. Демонстрация: состав архива

Собираем настройки из секретов (Блок 0b) и загружаем архив (Блок 1),
печатаем перечень документов.

In [ ]:
settings = Settings(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    model=OPENAI_MODEL,
    llm_provider_only=LLM_PROVIDER_ONLY,
)

archive = Archive(DATA_DIR)
docs = archive.documents()
print(f"В архиве {len(docs)} документов:\n")
print(archive.list_text())


## Блок 8. Демонстрация: запросы к ассистенту

Каждый запрос проходит весь конвейер (стадии 1–3). В выводе видно прогресс:
черновик, итерации цикла инструментов и вызовы функций. Финальный ответ выводится
в Markdown со ссылками на источники `[doc_id:N, page:M]`.

Экономия: по умолчанию выполняется только первый вопрос (`QUERIES[:1]`).
Чтобы прогнать все — замените на `QUERIES` (или `QUERIES[:2]`).

In [ ]:
from IPython.display import Markdown, display

assistant = MiniAssistant(archive, settings)

QUERIES = [
    "Классификация судов по назначению: какие классы существуют и что они означают?",
    "Какие требования предъявляются к противопожарной защите судов?",
    "Что входит в механические установки судна?",
]

# Сколько запросов прогнать: QUERIES[:1] — экономично, QUERIES — полная демонстрация.
DEMO_QUERIES = QUERIES[:1]

for q in DEMO_QUERIES:
    print("\n" + "=" * 90, flush=True)
    print("ВОПРОС:", q, flush=True)
    print("=" * 90, flush=True)
    try:
        answer = assistant.run(q)
        display(Markdown(answer))
    except Exception as exc:
        print(f"Ошибка при обработке запроса: {exc}")
